# Pydantic v2 models, validators, JSON schema generation

*0.1 Python for GenAI · run **Setup** first*

## Setup

Settings, a configured client, and two helpers. Every cell below uses them.

In [1]:
"""Shared setup for this notebook: typed settings, configured clients, logging."""

import asyncio
import json
import logging
from concurrent.futures import ThreadPoolExecutor

from dotenv import find_dotenv
from openai import AsyncOpenAI, OpenAI
from pydantic import Field, SecretStr
from pydantic_settings import BaseSettings, SettingsConfigDict


class Settings(BaseSettings):
    """All configuration in one validated object, read from the environment / .env."""

    model_config = SettingsConfigDict(env_file=find_dotenv(), extra="ignore")

    openai_api_key: SecretStr
    openai_model: str = "gpt-4o-mini"
    request_timeout_seconds: float = Field(default=30, gt=0)
    max_retries: int = Field(default=2, ge=0, le=5)


settings = Settings()

client = OpenAI(
    api_key=settings.openai_api_key.get_secret_value(),
    timeout=settings.request_timeout_seconds,
    max_retries=settings.max_retries,
)


def async_client() -> AsyncOpenAI:
    """A fresh async client per event loop (async clients are bound to the loop they run in)."""
    return AsyncOpenAI(
        api_key=settings.openai_api_key.get_secret_value(),
        timeout=settings.request_timeout_seconds,
        max_retries=settings.max_retries,
    )


def run_async(coroutine):
    """Run a coroutine from a notebook (which already has an event loop). Scripts use asyncio.run()."""
    with ThreadPoolExecutor(max_workers=1) as pool:
        return pool.submit(asyncio.run, coroutine).result()


def show(title: str, value) -> None:
    """Print a labelled, formatted JSON block."""
    print(title)
    print(json.dumps(value, indent=2, ensure_ascii=False, default=str))


logging.basicConfig(level=logging.WARNING, format="%(levelname)s %(name)s: %(message)s")
for noisy in ["httpx", "httpx2", "httpcore", "openai"]:
    logging.getLogger(noisy).setLevel(logging.WARNING)

print("model:", settings.openai_model, "| timeout:", settings.request_timeout_seconds, "s")

model: gpt-4o-mini | timeout: 30.0 s


### Pydantic v2 models

> **Problem.** A signup form sends `{"age": "thirty", "email": "priya"}`. The code stores it. Three weeks later a report crashes on `age + 1`, and a mail job bounces on the bad email — two bugs, far from where the bad data entered, with no clue which request caused them.

**Idea.** Describe the correct shape once as a class; every record is checked against it at the door, and every mistake is reported by field.

**Use when** at every boundary: API bodies, queue messages, model output, config.  
**Not when** inside hot loops on data you already trust — use dataclasses.

```mermaid
flowchart LR
    D[incoming data] --> V{Pydantic · validate}
    V -->|good| O[typed object]
    V -->|bad| E["errors by field: · email · age"]
```

**How it works.**
1. `class Customer(BaseModel)` lists the fields with their types: `name: str`, `age: int`, `email: EmailStr`, a nested `Address`.
2. `Customer.model_validate(data)` walks the dictionary field by field, converting where safe (`"42"` → `42`) and filling defaults.
3. If anything fails, it does not stop at the first problem: it collects every error and raises one `ValidationError`.
4. `error.errors()` is a list — each entry names the field path and what was wrong. That list goes back to the caller.
5. On success you hold a `Customer` object; every later line of code can trust its fields and types.

| | what happens | result |
|:--|:--|:--|
| ✓ good input | `{"age": "30"}` | `Customer(age=30)` — coerced |
| ✗ bad input | `{"age": "thirty", "email": "nope"}` | 4 errors, each naming its field |

**Production code and its real output**

In [2]:
# Pydantic models — the request/response types a service defines once and reuses for API input,
# storage rows and LLM structured output. Parsing coerces, fills defaults, and reports every error.
from pydantic import BaseModel, EmailStr, ValidationError


class Address(BaseModel):
    city: str
    country: str = "India"


class Customer(BaseModel):
    customer_id: int
    name: str = Field(min_length=1, max_length=80)
    email: EmailStr
    address: Address
    tags: list[str] = Field(default_factory=list)


customer = Customer.model_validate(
    {
        "customer_id": "42",
        "name": "Acme",
        "email": "ops@acme.example",
        "address": {"city": "Chennai"},
    }
)
show("parsed", customer.model_dump())

try:
    Customer.model_validate({"customer_id": "x", "name": "", "email": "nope", "address": {}})
except ValidationError as error:
    problems = []
    for item in error.errors():
        problems.append({"field": ".".join(map(str, item["loc"])), "msg": item["msg"]})
    show("rejected", problems)

assert customer.customer_id == 42 and len(problems) == 4

parsed
{
  "customer_id": 42,
  "name": "Acme",
  "email": "ops@acme.example",
  "address": {
    "city": "Chennai",
    "country": "India"
  },
  "tags": []
}
rejected
[
  {
    "field": "customer_id",
    "msg": "Input should be a valid integer, unable to parse string as an integer"
  },
  {
    "field": "name",
    "msg": "String should have at least 1 character"
  },
  {
    "field": "email",
    "msg": "value is not a valid email address: An email address must have an @-sign."
  },
  {
    "field": "address.city",
    "msg": "Field required"
  }
]


**What the output shows.** The good record came back typed and normalised; the bad one produced four errors in one go — `customer_id`, `name`, `email`, `address.city` — exactly what a 422 response should contain.

**In practice**
- **boundaries only** — validate where data enters (HTTP body, queue, LLM output, config). Re-validating trusted objects in inner loops wastes CPU for nothing.
- **extra="forbid"** — by default unknown fields are silently dropped, so a client sending `emial` never learns it is wrong. Forbid extras on inbound models.
- **strict when it matters** — `"30"` → `30` is convenient for forms and dangerous for money; `strict=True` on those models.
- **mutable defaults** — `tags: list = []` is shared by every object; use `Field(default_factory=list)`.
- **versioning** — models stored in a database or queue outlive the code; add fields with defaults, never rename without a migration.

**Alternatives** — dataclasses (no validation, faster) · `attrs` · marshmallow (older, more ceremony)

**Terms** — *validation*: checking data against rules · *coercion*: quiet conversion, `"30"` → `30` · *422*: the HTTP reply for "your input failed validation"


### validators

> **Problem.** The invoice rule "total = subtotal × (1 + tax)" is checked in the web form, again in the import script, and again in the LLM extraction path. Someone fixes a rounding bug in one of the three. A month later the other two disagree with it, and finance asks which number is right.

**Idea.** Attach business rules to the model, so every code path enforces them identically.

**Use when** a rule must hold wherever the data appears.  
**Not when** the rule needs a database or a network call — that belongs in a service layer.

```mermaid
flowchart LR
    F[fields set] --> FV["@field_validator · one field"] --> MV["@model_validator(after) · fields together"] --> OK[accepted]
    FV -->|bad| X[ValidationError]
    MV -->|bad| X
```

**How it works.**
1. `@field_validator("customer")` runs on that one field: it can clean the value (`"  acme corp "` → `"Acme Corp"`) or reject it with `raise ValueError(...)`.
2. Field validators run first, one field at a time, so by the time the model validator runs every field is already valid on its own.
3. `@model_validator(mode="after")` runs on the finished object and can compare fields: it recomputes the expected total and rejects a mismatch.
4. A `ValueError` raised inside either becomes a normal `ValidationError` with the field name attached — callers see one consistent error format.
5. Because the rules live on the class, the form, the import script and the LLM path all get them for free.

| | what happens | result |
|:--|:--|:--|
| ✓ | `subtotal=100, tax=0.18, total=118` | accepted; customer normalised to "Acme Corp" |
| ✗ | `total=999` | "total 999 != 118" |
| ✗ | `customer="   "` | "customer must not be blank" |

**Production code and its real output**

In [3]:
# Validators — business rules that live on the model, so every code path enforces the same rules.
from pydantic import BaseModel, ValidationError, field_validator, model_validator


class Invoice(BaseModel):
    invoice_id: str = Field(pattern=r"^INV-\d{4}$")
    customer: str
    subtotal: float = Field(ge=0)
    tax_rate: float = Field(ge=0, le=0.5)
    total: float

    @field_validator("customer")
    @classmethod
    def normalise_customer(cls, value: str) -> str:
        if not value.strip():
            raise ValueError("customer must not be blank")
        return value.strip().title()

    @model_validator(mode="after")
    def total_must_match(self) -> "Invoice":
        expected = round(self.subtotal * (1 + self.tax_rate), 2)
        if abs(self.total - expected) > 0.01:
            raise ValueError(f"total {self.total} != {expected}")
        return self


valid = Invoice(
    invoice_id="INV-0042", customer="  acme corp ", subtotal=100, tax_rate=0.18, total=118
)
print("valid:", valid.customer, valid.total)

bad_inputs = {
    "bad id": {
        "invoice_id": "42",
        "customer": "Acme",
        "subtotal": 100,
        "tax_rate": 0.18,
        "total": 118,
    },
    "blank customer": {
        "invoice_id": "INV-0001",
        "customer": " ",
        "subtotal": 1,
        "tax_rate": 0,
        "total": 1,
    },
    "wrong total": {
        "invoice_id": "INV-0001",
        "customer": "Acme",
        "subtotal": 100,
        "tax_rate": 0.18,
        "total": 999,
    },
}
for label, data in bad_inputs.items():
    try:
        Invoice.model_validate(data)
    except ValidationError as error:
        print(f"{label:<15} -> {error.errors()[0]['msg']}")

assert valid.customer == "Acme Corp"

valid: Acme Corp 118.0
bad id          -> String should match pattern '^INV-\d{4}$'
blank customer  -> Value error, customer must not be blank
wrong total     -> Value error, total 999.0 != 118.0


**What the output shows.** The valid invoice came back with the customer name normalised; each of the three bad inputs was rejected with a message naming the exact rule it broke.

**In practice**
- **pure** — no database or network inside a validator; they run on every object creation, so an I/O call becomes thousands of calls and a flaky test suite.
- **messages** — the message is what the API caller or the on-call engineer reads — say what was expected, not just that it failed.
- **placement** — a rule about two fields goes in `model_validator`; a field validator that peeks at other fields breaks when field order changes.
- **normalise, don't change meaning** — trimming and title-casing a name is fine; rewriting a product code is a silent data change nobody asked for.
- **test per rule** — each validator gets its own small test with a passing and a failing example; they are the executable spec of the business rule.

**Alternatives** — a separate validation function called explicitly (easy to forget) · database constraints for storage rules (last line of defence)

**Terms** — *field validator*: cleans or rejects one field · *model validator*: checks fields against each other · *normalise*: bring to one standard form


### JSON schema generation

> **Problem.** You ask the model to "extract the invoice as JSON". Nine times out of ten you get JSON. The tenth time you get "Sure! Here is the invoice:" followed by JSON, or a field named `Total` instead of `total`, or a number written as `"1,250.50"`. Your parser breaks, at 2% of requests, forever.

**Idea.** Send the model's JSON schema with the request; the provider then only produces output in that exact shape.

**Use when** model output feeds code: extraction, routing, tool arguments.  
**Not when** free-form text is the product (a chat reply, an essay).

```mermaid
flowchart LR
    M[Pydantic model] --> S[JSON schema] --> A[LLM API · constrained decoding] --> O[parsed object]
```

**How it works.**
1. `Invoice.model_json_schema()` turns the class into a JSON schema: field names, types, which are required, nested `LineItem`s.
2. `client.beta.chat.completions.parse(..., response_format=Invoice)` sends that schema with the request.
3. On the provider's side, decoding is constrained: at each step only tokens that keep the output valid under the schema are allowed. A stray sentence is impossible.
4. The SDK parses the reply straight into an `Invoice` object and validates it — `.parsed` is typed, not text.
5. If the model cannot satisfy the schema (rare), you get a refusal or an error, not broken JSON.

| | what happens | result |
|:--|:--|:--|
| ✗ plain prompt | "reply as JSON" | usually JSON; sometimes a sentence first — code breaks |
| ✓ schema | `response_format=Invoice` | `Invoice(invoice_id='INV-0077', items=[…], total=55.0)` every time |

**Production code and its real output**

In [4]:
# JSON schema generation — the model's schema goes to the API as a strict response format, so the
# reply is guaranteed to match it. No hand-written parsing.
from pydantic import BaseModel


class LineItem(BaseModel):
    description: str
    quantity: int = Field(ge=1)
    unit_price: float = Field(ge=0)


class Invoice(BaseModel):
    invoice_id: str = Field(pattern=r"^INV-\d{4}$", description="Format INV-0000")
    customer: str
    items: list[LineItem]
    total: float


schema = Invoice.model_json_schema()
show(
    "schema (top level)", {"required": schema["required"], "properties": list(schema["properties"])}
)

text = "Invoice INV-0077 for Globex: 3 x widget at 10.00, 1 x gadget at 25.00, total 55.00"
response = client.beta.chat.completions.parse(
    model=settings.openai_model,
    messages=[{"role": "user", "content": f"Extract the invoice: {text}"}],
    response_format=Invoice,
    temperature=0,
)
invoice = response.choices[0].message.parsed
show("extracted", invoice.model_dump())
assert invoice.invoice_id == "INV-0077" and len(invoice.items) == 2

schema (top level)
{
  "required": [
    "invoice_id",
    "customer",
    "items",
    "total"
  ],
  "properties": [
    "invoice_id",
    "customer",
    "items",
    "total"
  ]
}


extracted
{
  "invoice_id": "INV-0077",
  "customer": "Globex",
  "items": [
    {
      "description": "widget",
      "quantity": 3,
      "unit_price": 10.0
    },
    {
      "description": "gadget",
      "quantity": 1,
      "unit_price": 25.0
    }
  ],
  "total": 55.0
}


**What the output shows.** The schema lists exactly four required fields and a nested `LineItem` definition; the reply came back already parsed into an `Invoice` with two line items and total 55.0.

**In practice**
- **descriptions** — the model reads each field's `description` to decide what goes there; a field without one gets guessed.
- **optional fields** — a required field the model cannot know from the text will be invented; make it `Optional` and check for `None`.
- **flat schemas** — strict mode rejects some constructs (defaults, some unions); deep nesting costs tokens and accuracy. Keep it small.
- **shape ≠ truth** — a valid `total: 55.0` can still be wrong; validate values with validators or a second check.
- **version it** — changing a field name changes what the model produces; treat the schema like an API contract.

**Alternatives** — JSON mode (valid JSON, no schema) · Instructor (retries on validation failure, any provider) · grammar-constrained decoding for local models (Outlines, llama.cpp GBNF)

**Terms** — *JSON*: data as text: `{"total": 55}` · *schema*: formal description of a JSON shape · *structured output*: forcing the model to follow a schema
